# Playground
**Nov 16th 2025**
**Jakob Balkovec**

Testing out how the imputers perform.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while root.name != "Pipeline" and root.parent != root:
    root = root.parent

if root.name != "Pipeline":
    raise RuntimeError("pipeline dir not found")

sys.path.append(str(root))

import pandas as pd
import json
from MDR.Temporal.Pipeline.deprecated.imputers import transform_with_ensemble

In [9]:
PATH = r'/Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/experiments/missing_values/satellite/satellite_data_darrington_2022_2024.csv'
df = pd.read_csv(PATH)

In [18]:
def get_season_mask(df, season):
    month = df['date'].dt.month

    if season == "winter":
        return month.isin([12, 1, 2])
    if season == "spring":
        return month.isin([3, 4, 5])
    if season == "summer":
        return month.isin([6, 7, 8])
    if season == "fall":
        return month.isin([9, 10, 11])

    raise ValueError("season must be winter, spring, summer, or fall")

## Overall Performance

In [20]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

col = 'NDVI'
col2 = 'LST'

out_ndvi, diag_ndvi = transform_with_ensemble(df, col, return_diag=True)
out2_lst, diag2_lst = transform_with_ensemble(df, col2, return_diag=True)

Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|███████

In [21]:
print(json.dumps(diag_ndvi, indent=4))

{
    "feature": "NDVI",
    "n_missing": 1027,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 1014,
            "avg_conf": 0.10298936042284804,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 1014,
            "avg_conf": 0.3102886481798984,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 411,
            "avg_conf": 0.14285714285714282,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 0,
            "avg_conf": 0.0,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 1027,
            "avg_conf": 1.0,
          

In [22]:
print(json.dumps(diag2_lst, indent=4))

{
    "feature": "LST",
    "n_missing": 682,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 674,
            "avg_conf": 0.36544629555289476,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 674,
            "avg_conf": 0.5077583234396958,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 538,
            "avg_conf": 0.31253319171534777,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 382,
            "avg_conf": 0.12356020942408376,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 682,
            "avg_conf": 1.

## Seasons

### Summer

_Referencing the data below_

#### NDVI

For summer, NDVI leaned heavily on the ML models. The classical imputers helped, but they definitely weren’t driving the bus.

**Standouts**
- Linear Model: Used every missing point. Confidence was basically perfect. It’s the clearest signal NDVI has in this window.

- XGBoost: Also stepped up for every gap with strong confidence. Solid second-best performer.

**Middle of the pack**
- ffill/bfill: Covered almost everything, confidence around one-third. Useful, but not something we want as our primary.
- Linear Interpolation: Same coverage as ffill, but confidence was low. It fills gaps, but not with much certainty.
- Rolling Mean: Much more selective. Lower coverage and modest confidence. Works, but only in certain stretches.

**Non-impact player**
- Climatology: Stayed active but didn’t contribute anything meaningful. NDVI seasonality just wasn’t strong enough here...

**Ranking**
| Rank | Strategy      |
|------|---------------|
| 1    | Linear Model  |
| 2    | XGBoost       |
| 3    | ffill/bfill   |
| 4    | Linear Interp |
| 5    | Rolling Mean  |
| 6    | Climatology   |
#### LST

What happened

LST behaved more predictably. Even the simpler imputers performed better, and the ML models were steady.

Standouts
- Linear Model: Perfect coverage, perfect confidence. Best model in this season by a wide margin.	
- XGBoost: Also fully engaged with strong confidence. Consistent and reliable.

Middle of the Pack
- ffill/bfill: Covered almost everything with solid confidence. Makes sense...LST usually doesn’t jump wildly day to day.
- Linear Interp: Similar coverage with lower confidence. Good when gaps are small.

Moderate performer
- Rolling Mean: Lower coverage and moderate confidence. More smoothing than true prediction.

Low-impact
- Climatology: Contributed, but confidence was low. Seasonal structure was weaker here.

**Ranking**
| Rank | Strategy      |
|------|---------------|
| 1    | Linear Model  |
| 2    | XGBoost       |
| 3    | ffill/bfill   |
| 4    | Linear Interp |
| 5    | Rolling Mean  |
| 6    | Climatology   |

In [27]:
season = "summer"
mask = get_season_mask(df, season)
df_season = df[mask].copy()

out_summer_ndvi, diag_summer_ndvi = transform_with_ensemble(
    df_season,
    "NDVI",
    return_diag=True
)

out_summer_lst, diag_summer_lst = transform_with_ensemble(
    df_season,
    "LST",
    return_diag=True
)

Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
not enough samples (18) to train imputer -- DISABLING XGB
imputer 'xgboost' disabled during fit
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading confi

In [28]:
print(json.dumps(diag_summer_ndvi, indent=4))

{
    "feature": "NDVI",
    "n_missing": 258,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 246,
            "avg_conf": 0.09301956613191481,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 246,
            "avg_conf": 0.31192826914226185,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 108,
            "avg_conf": 0.1428571428571428,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 0,
            "avg_conf": 0.0,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 258,
            "avg_conf": 1.0,
            "s

In [29]:
print(json.dumps(diag_summer_lst, indent=4))

{
    "feature": "LST",
    "n_missing": 118,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 112,
            "avg_conf": 0.45685872788962906,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 112,
            "avg_conf": 0.5885815850317998,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 106,
            "avg_conf": 0.40161725067385445,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 100,
            "avg_conf": 0.141,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 118,
            "avg_conf": 1.0,
           

### Spring

_Referencing the data below_

#### NDVI

Quick take: Linear Model absolutely carries. XGBoost didn't do that good.

What happened:
- Linear Model hits 258/258 missing spots with 1.0 confidence. Full sweep. Spring NDVI is super predictable for it.
- XGBoost is inactive. It didn’t have enough known samples this season to train. Not surprising — spring NDVI is messy.
- ffill/bfill hits everything (246/258) with an okay confidence (~0.31). It’s mid, but at least it shows up.
- Linear Interp matches ffill’s coverage but confidence drops to ~0.09. Works, just weak.
- Rolling Mean barely shows up (42 percent coverage).
- Climatology does absolutely nothing. NDVI climatology sucks for this region

**Ranking**
| Rank | Strategy      |
|------|---------------|
| 1    | Linear Model  |
| 2    | ffill/bfill   |
| 3    | Linear Interp |
| 4    | Rolling Mean  |
| 5    | Climatology   |
| 6    | XGBoost       |

#### LST

Quick take: Everything works here. This is the cleanest season for LST. Linear Model and XGB are monsters.

What happened:
- Linear Model perfect again: 100 percent coverage, 1.0 confidence.
- XGBoost also perfect: 189/189, 0.70 confidence. Very solid.
- ffill/bfill shows great instincts in spring: 186/189, strong confidence (~0.50). Better than I expected.
- Linear Interp does similar coverage with weaker confidence (~0.32).
- Rolling Mean contributes a lot (149/189), but confidence is mixed.
- Climatology actually wakes up for LST: 109 contributions, even though the confidence is low. I guess, it’s finally doing something.

**Ranking**
| Rank | Strategy      |
|------|---------------|
| 1    | Linear Model  |
| 2    | XGBoost       |
| 3    | ffill/bfill   |
| 4    | Linear Interp |
| 5    | Rolling Mean  |
| 6    | Climatology   |


In [31]:
season = "spring"
mask = get_season_mask(df, season)
df_season = df[mask].copy()

out_spring_ndvi, diag_spring_ndvi = transform_with_ensemble(
    df_season,
    "NDVI",
    return_diag=True
)

out_spring_lst, diag_spring_lst = transform_with_ensemble(
    df_season,
    "LST",
    return_diag=True
)

Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
not enough samples (18) to train imputer -- DISABLING XGB
imputer 'xgboost' disabled during fit
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading confi

In [32]:
print(json.dumps(diag_spring_ndvi, indent=4))

{
    "feature": "NDVI",
    "n_missing": 258,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 246,
            "avg_conf": 0.09301956613191481,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 246,
            "avg_conf": 0.31627432952791074,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 108,
            "avg_conf": 0.1428571428571428,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 0,
            "avg_conf": 0.0,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 258,
            "avg_conf": 1.0,
            "s

In [33]:
print(json.dumps(diag_spring_lst, indent=4))

{
    "feature": "LST",
    "n_missing": 189,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 186,
            "avg_conf": 0.32391344304685754,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 186,
            "avg_conf": 0.49576957163697144,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 149,
            "avg_conf": 0.28667305848513897,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 109,
            "avg_conf": 0.11376146788990826,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 189,
            "avg_conf": 1

### Fall

_Referencing the data below_

#### NDVI

Quick take: Linear Model carries again, XGB naps again, and everything else chips in with medium-to-low confidence.

What happened:
- Linear Model sweeps the board: 258/258, 1.0 confidence. Absolute anchor.
- XGBoost is inactive again. Fall NDVI doesn’t give it enough clean samples to train.
- ffill/bfill does mid work: 231/258, decent confidence (~0.28).
- Linear Interp mirrors ffill’s coverage but confidence is weak (~0.08).
- Rolling Mean only helps in 90 cases. Same low confidence as usual.
- Climatology contributes nothing. Fall NDVI is too variable for seasonal averaging.

**Ranking**
| Rank | Strategy      |
|------|---------------|
| 1    | Linear Model  |
| 2    | ffill/bfill   |
| 3    | Linear Interp |
| 4    | Rolling Mean  |
| 5    | Climatology   |
| 6    | XGBoost       |

#### LST

Quick take: Same story as spring. LST is clean, and all the models behave. Linear Model and XGB crush, ffill/bfill is reliably good, and the rest fill in the gaps.

What happened:
- Linear Model perfect: 169/169, 1.0 confidence. No surprises.
- XGBoost also perfect: 169/169, 0.70 confidence. Strong.
- ffill/bfill does very well: 169/169, confidence just under 0.50.
- Linear Interp matches full coverage with slightly lower confidence (~0.35).
- Rolling Mean contributes ~75 percent of the missing values (127/169), solid.
- Climatology wakes up a bit: 91 contributions, but confidence is low (~0.12).

**Ranking**
| Rank | Strategy      |
|------|---------------|
| 1    | Linear Model  |
| 2    | XGBoost       |
| 3    | ffill/bfill   |
| 4    | Linear Interp |
| 5    | Rolling Mean  |
| 6    | Climatology   |

In [34]:
season = "fall"
mask = get_season_mask(df, season)
df_season = df[mask].copy()

out_fall_ndvi, diag_fall_ndvi = transform_with_ensemble(
    df_season,
    "NDVI",
    return_diag=True
)

out_fall_lst, diag_fall_lst = transform_with_ensemble(
    df_season,
    "LST",
    return_diag=True
)

Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
not enough samples (15) to train imputer -- DISABLING XGB
imputer 'xgboost' disabled during fit
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading confi

In [35]:
print(json.dumps(diag_fall_ndvi, indent=4))

{
    "feature": "NDVI",
    "n_missing": 258,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 231,
            "avg_conf": 0.07924783815913779,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 231,
            "avg_conf": 0.28390250865937977,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 90,
            "avg_conf": 0.1428571428571428,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 0,
            "avg_conf": 0.0,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 258,
            "avg_conf": 1.0,
            "sk

In [36]:
print(json.dumps(diag_fall_lst, indent=4))

{
    "feature": "LST",
    "n_missing": 169,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 169,
            "avg_conf": 0.351841247661223,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 169,
            "avg_conf": 0.4908146506677329,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 127,
            "avg_conf": 0.33970753655793046,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 91,
            "avg_conf": 0.12087912087912087,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 169,
            "avg_conf": 1.0,


### Winter

_Referencing the data below_

#### NDVI

Quick take: This is basically the spring/fall NDVI situation copy-pasted again. Nothing dramatic changes in winter except that vegetation goes dark and XGB gives up completely.

What happened:
- Linear Model is once again the undisputed champ. Full coverage: 253/253, confidence 1.0. Winter NDVI is linear-model heaven because the trend is dead simple.
- XGBoost stays inactive. Winter NDVI is too flat and too sparse to train a nontrivial model.
- ffill/bfill still useful: 240 contributions, confidence around 0.31.
- Linear Interp matches ffill coverage with weaker confidence (~0.09).
- Rolling Mean helps in 105 cases. Confidence unchanged (~0.14).
- Climatology again contributes nothing. Seasonal NDVI patterns in winter are basically a flatline.

**Ranking**
| Rank | Strategy      |
|------|---------------|
| 1    | Linear Model  |
| 2    | ffill/bfill   |
| 3    | Linear Interp |
| 4    | Rolling Mean  |
| 5    | Climatology   |
| 6    | XGBoost       |

#### LST

Quick Take: Winter LST is easy mode. All models activate, all models contribute, and the ranking is consistent with the yearly pattern.

What happened:
- Linear Model is the top dog again. Full coverage 206/206, confidence 1.0. LST is smooth in winter, so the linear model nails it.
- XGBoost is healthy and performs very well. Same full coverage, confidence 0.70 which is excellent for a seasonal model.
- ffill/bfill is super dependable. 198 contributions with confidence around 0.48. The thermal signal changes slowly in winter, so ffill/bfill is very reasonable.
- Linear Interpolation also hits 198 contributions, mediocre confidence (~0.30). Still a useful middle-of-the-pack model.
- Rolling Mean steps up with 155 contributions, confidence 0.25. Good smoothing for cold-season temperature gaps.
- Climatology gives 82 contributions, confidence 0.11. Winter climatology for LST isn’t terrible, but it’s clearly not a top-tier contributor.

**Ranking**
| Rank | Strategy        |
|------|-----------------|
| 1    | Linear Model    |
| 2    | XGBoost         |
| 3    | ffill/bfill     |
| 4    | Linear Interp   |
| 5    | Rolling Mean    |
| 6    | Climatology     |

In [37]:
season = "winter"
mask = get_season_mask(df, season)
df_season = df[mask].copy()

out_winter_ndvi, diag_winter_ndvi = transform_with_ensemble(
    df_season,
    "NDVI",
    return_diag=True
)

out_winter_lst, diag_winter_lst = transform_with_ensemble(
    df_season,
    "LST",
    return_diag=True
)

Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
not enough samples (18) to train imputer -- DISABLING XGB
imputer 'xgboost' disabled during fit
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading config: 100%|██████████████████████████████████████████████████████| 3/3
Loading confi

In [38]:
print(json.dumps(diag_winter_ndvi, indent=4))

{
    "feature": "NDVI",
    "n_missing": 253,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 240,
            "avg_conf": 0.09188784875976724,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 240,
            "avg_conf": 0.31197325222007855,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 105,
            "avg_conf": 0.1428571428571428,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 0,
            "avg_conf": 0.0,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 253,
            "avg_conf": 1.0,
            "s

In [40]:
print(json.dumps(diag_winter_lst, indent=4))

{
    "feature": "LST",
    "n_missing": 206,
    "imputers": {
        "linear_interp": {
            "active": true,
            "contributions": 198,
            "avg_conf": 0.29552646046319475,
            "skipped": false,
            "fail_reason": null
        },
        "ffill_bfill": {
            "active": true,
            "contributions": 198,
            "avg_conf": 0.47857111331409574,
            "skipped": false,
            "fail_reason": null
        },
        "rolling_mean": {
            "active": true,
            "contributions": 155,
            "avg_conf": 0.2497695852534562,
            "skipped": false,
            "fail_reason": null
        },
        "climatology": {
            "active": true,
            "contributions": 82,
            "avg_conf": 0.11585365853658537,
            "skipped": false,
            "fail_reason": null
        },
        "linear_model": {
            "active": true,
            "contributions": 206,
            "avg_conf": 1.0

## Structure

Think of this as a top-level overview of the system...

The system fills missing satellite data by running several independent imputers and then blending their predictions with a weighted voting scheme. 

Each imputer captures a different assumption about how environmental signals behave. The `VotingImputer` fuses their outputs, down-weights outliers, and produces both a final value and a confidence score. Models can also disable themselves automatically if training is impossible.

**Weights**:
```yaml
  base_weights:
    linear_model: 0.40
    xgboost: 0.30
    ffill_bfill: 0.10
    linear_interp: 0.08
    rolling_mean: 0.07
    climatology: 0.05
```

These were adjusted after my tests. Since the linear model performed a lot better than climatology, its vote carries more weight. I hope that makes sense.

These weights will be adjusted as we get more data and more features.

## Imputer Strategies (Brief)

**Linear Interpolation**
- Connects the nearest valid points with a straight line. Works for small, smooth gaps. Confidence drops as gaps widen.

**Forward–Backward Fill**
- Carries the last or next valid value into the gap (zero-order hold). Stable for short gaps, oversmooths long ones.

**Rolling Mean**
- Uses a centered moving average. Good at smoothing medium gaps, but blurs peaks.

**Climatology**
- Fills using the long-term average for each day-of-year. Strong for long gaps if enough historical data exists.

**Linear Model**
- A regression model using seasonal encodings plus cross-feature predictors (LST, NDVI, Rain_sat). Very strong when enough training data is available.

**XGBoost**
- A nonlinear boosted tree model capturing complex feature interactions. High performance, but needs sufficient valid samples.

## Voting System (Brief)
1. Every active imputer predicts a value plus a confidence score
2. Each model’s influence is `weight = base_weight * confidence`
3. Outliers are identified using the median and MAD and down-weighted heavily
4. The final value and confidence are computed as weighted averages
5. Any imputer that errors, fails training, or provides invalid output is automatically disabled

## Why it Works

The fact that different models excel under different conditions (short gaps, long gaps, seasonal patterns, nonlinear structure) is very important here. 

The voting layer integrates all their strengths and protects against individual failures. The result is a stable, high-confidence reconstruction across seasons and sensors.

## Why it's Future Proof

We can drop in or remove any imputation strategy without breaking the pipeline. Every model fails gracefully with a logged warning, so the workflow never stalls...The ensemble also handles model selection implicitly. Even if one approach isn’t a good fit for the current feature or season, the voting layer absorbs that and shifts influence toward the methods that actually perform well.

Some potential additions:
- LOESS / LOWESS smoothing
- Kalman filter imputer
- Seasonal ARIMA / SARIMAX
- KNN imputer
- Gaussian process regression
- Spline interpolation (cubic or B-spline)
- Autoencoder reconstruction
- Temporal convolutional network imputer
- Transformer-based imputers (BRITS, GRIN, SAITS)

## Interpretation of the Results (General)

Across every season and for both NDVI and LST, the ensemble behaves in a stable and predictable way, that's good! 

The Linear Model is consistently the dominant contributor...it handles almost all missing values and assigns itself high confidence because the underlying temporal features (day of year, year, sine/cosine encodings) provide strong seasonal structure. 

XGBoost follows as the second-most influential model whenever it has enough data to train. Its confidence stays high, and the ensemble relies on it heavily for larger gaps or more irregular behavior.

The simpler statistical models play a support role. ffill/bfill and linear interpolation remain useful for short gaps, especially when the missing segments are surrounded by clean data. Their confidence values are moderate, which keeps them influential without letting them dominate the ensemble. 

Rolling Mean contributes mainly in noisier regions but with lower confidence, which is appropriate given that smoothing can blur actual temporal variation. 

Climatology tends to contribute almost nothing unless the seasonal signal is strong and well sampled. This is kind of expected, it’s a long-term average method that shines only when the other models lack usable information...

The ensemble’s behavior is robust...when a method fails to train (typically XGBoost in NDVI shoulder seasons) it is correctly marked inactive and removed from the vote. The remaining models absorb its role without destabilizing the final predictions. 

The overall pattern is pretty consistent: deterministic regressors dominate, lightweight methods fill small gaps, and seasonal averages matter only when everything else weakens.

---
**Jakob Balkovec**
